# HyperParameter Tuning

In [19]:
from scikeras.wrappers import KerasClassifier
import tensorflow
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping
import pandas as pd
import pickle
from sklearn.preprocessing import LabelEncoder,StandardScaler,OneHotEncoder
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline

In [20]:
df=pd.read_csv("C:\\Users\\ssan\\Downloads\\Cursor_Projects\\Deep_Learning\\Churn_Modelling.csv")
df.drop(['RowNumber','CustomerId','Surname'],axis=1,inplace=True)
labelencode= LabelEncoder()
df['Gender']= labelencode.fit_transform(df['Gender'])
onehotencoder=OneHotEncoder(handle_unknown='ignore')
onehotencode_geo= onehotencoder.fit_transform(df[['Geography']])
onehotencode_geo.toarray()
geo_df= pd.DataFrame(onehotencode_geo.toarray(), columns=onehotencoder.get_feature_names_out(['Geography']))
df=pd.concat([df.drop('Geography',axis=1),geo_df],axis=1).reset_index(drop=True)
X=df.drop('Exited',axis=1)
y=df['Exited']
X_train,X_test,y_train,y_test= train_test_split(X,y,test_size=0.20,random_state=77)

from sklearn.preprocessing import StandardScaler

scaler= StandardScaler()
X_train= scaler.fit_transform(X_train)
X_test= scaler.transform(X_test)
# # export all encoders 

# with open('label_encode_gender.pkl','wb') as file:
#     pickle.dump(label_encode_gender,file)

# with open('onehotencode.pkl','wb') as file:
#     pickle.dump(encoder_geo,file)

# with open('scaler.pkl','wb') as file:
#     pickle.dump(scaler,file)


In [21]:
df.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [22]:
## Create a model

def create_model(neurons=32, layers=1):
    model=Sequential()
    model.add(Dense(neurons,activation='relu',input_shape=(X_train.shape[1],)))

    for _ in range(layers-1):
        model.add(Dense(neurons,activation='relu'))

    model.add(Dense(1,activation='sigmoid'))
    model.compile(optimizer='adam',loss='binary_crossentropy',metrics=['accuracy'])
    return model

In [23]:
model=KerasClassifier(neurons=32,layers=1,build_fn=create_model,batch_size=10,epochs=50,verbose=1)

In [24]:
params={
    'neurons': [64,32,16,128],
    'layers': [1,2,3],
    'epochs': [10,50,100]
}

In [25]:
grid_cv= GridSearchCV(estimator=model,param_grid=params, n_jobs=-1,cv=3)
grid_val=grid_cv.fit(X_train,y_train)

KeyboardInterrupt: 

In [26]:
print("The best value: %f and its params : %f" % (grid_cv.best_score_, grid_cv.best_params_)) 

AttributeError: 'GridSearchCV' object has no attribute 'best_score_'